In [3]:
import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder 
                      .appName("PgSparkAidarAI") 
                      .config("spark.jars.packages",
                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
                        "org.postgresql:postgresql:42.5.0,"
                        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
                        ) 
                       .getOrCreate()
    )


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

In [ ]:
# spark.stop()

In [4]:
# ⬇️ Параметры подключения к PostgreSQL public.shops 

# тут задаем имя БД, в нашем случае это source

jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD")

table_shops = "public.shops"

shops_df = (spark.read.format("jdbc")
		.option("url", jdbc_url)
		.option("user", db_user)
		.option("password", db_password)
		.option("dbtable", table_shops)
		.option("fetchsize", 1000)
		.option("driver", "org.postgresql.Driver")
		.load() 
)

shops_df.show()

+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
|  847|      Diksi|
|  848|      Lenta|
|  849|   FixPrice|
|  850|     Magnit|
|  851|      Lenta|
+-----+-----------+



In [7]:
shops_df.printSchema()

root
 |-- st_id: integer (nullable = true)
 |-- shop_name: string (nullable = true)



In [5]:
table_shop_timezone = "public.shop_timezone" 

shop_timezone_df = (spark.read.format("jdbc")
		.option("url", jdbc_url)
		.option("user", db_user)
		.option("password", db_password)
		.option("dbtable", table_shop_timezone)
		.option("fetchsize", 1000)
		.option("driver", "org.postgresql.Driver")
		.load() 
)

shop_timezone_df.show()

+-----+---------+
|plant|time_zone|
+-----+---------+
|  842|         |
|  842|    RUS07|
|  843|    RUS04|
|  844|         |
|  845|         |
|  845|    RUS05|
|  847|    RUS03|
|  848|    RUS08|
|  848|         |
| P847|         |
| E103|    RUS08|
| -134|    RUS04|
|    0|         |
|    0|    RUS08|
|  848|         |
+-----+---------+



In [6]:
shop_timezone_df.printSchema()

root
 |-- plant: string (nullable = true)
 |-- time_zone: string (nullable = true)



In [13]:
# Регистрируем DataFrame-ы как временные вьюхи
shops_df.createOrReplaceTempView("shops")
shop_timezone_df.createOrReplaceTempView("shop_timezone")

In [18]:
spark.sql("""
    select
        s.st_id      as st_id    ,
        s.shop_name  as shop_name,
        case when st.time_zone is null or st.time_zone='' then 3 else cast(right(st.time_zone, 2) as int) end as tz_code
from
        shops s
join
        (select st.plant as plant, max(st.time_zone) as time_zone from shop_timezone st group by st.plant) st
on
        cast(s.st_id as string) = st.plant
""").show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  843|     Magnit|      4|
|  842|      Lenta|      7|
|  845|Pyaterochka|      5|
|  844|       Spar|      3|
|  848|      Lenta|      8|
|  847|      Diksi|      3|
+-----+-----------+-------+



In [17]:
spark.sql("""
    select
        s.st_id      as st_id    ,
        s.shop_name  as shop_name,
        case when st.time_zone is null or st.time_zone='' then 3 else cast(right(st.time_zone, 2) as int) end as tz_code
from
        shops s
join
        (select st.plant as plant, max(st.time_zone) as time_zone from shop_timezone st group by st.plant) st
on
        s.st_id= st.plant
order by s.st_id
""").show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+



In [19]:
result = spark.sql("""
    select
        s.st_id      as st_id    ,
        s.shop_name  as shop_name,
        case when st.time_zone is null or st.time_zone='' then 3 else cast(right(st.time_zone, 2) as int) end as tz_code
from
        shops s
join
        (select st.plant as plant, max(st.time_zone) as time_zone from shop_timezone st group by st.plant) st
on
        s.st_id= st.plant
order by s.st_id
""")

In [20]:
result.show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+



In [21]:
result.select('st_id','shop_name').where(''' shop_name == 'Lenta' ''').show()

+-----+---------+
|st_id|shop_name|
+-----+---------+
|  842|    Lenta|
|  848|    Lenta|
+-----+---------+



In [22]:
import pyspark.sql.functions as F

result.where(F.col('st_id') > 844).show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  845|Pyaterochka|      5|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+



In [23]:
# тормозим спарк
spark.stop()